# FlowEdit Tuned Bridge Evaluation Colab

Evaluate the FlowEdit released dataset with one FlowEdit baseline and several conservative `bridge_interpolate` settings. The defaults are chosen for a fair same-NFE comparison and to reduce the structure drift seen in the previous run.


## Setup

In [ ]:
# Configuration
import json
import os
from getpass import getpass
from pathlib import Path

REPO_URL = "https://github.com/Jiaqi-Ye/FlowEdit.git"
BRANCH = "flowedit-full-eval"
WORKDIR = "/content/FlowEdit"

model_name = "sd3"  # "sd3" or "flux"
methods_to_run = ["flowedit_baseline", "bridge_interpolate"]
budget_mode = "same_nfe"

# Tuned run: lower target CFG plus conservative bridge settings.
src_guidance_scale = None  # None keeps the model default: SD3=3.5, FLUX=1.5.
tar_guidance_scale = 10.5
bridge_settings = [
    {"setting_id": "l025_g200_w050", "pc_guidance_lambda": 0.25, "pc_guidance_gamma": 2.0, "pc_guidance_weight": 0.50},
    {"setting_id": "l050_g200_w050", "pc_guidance_lambda": 0.50, "pc_guidance_gamma": 2.0, "pc_guidance_weight": 0.50},
    {"setting_id": "l050_g200_w075", "pc_guidance_lambda": 0.50, "pc_guidance_gamma": 2.0, "pc_guidance_weight": 0.75},
]

DATASET_YAML = "Data/flowedit.yaml"
EXPERIMENT_TAG = f"flowedit_eval_tuned_{model_name}_same_nfe_tar{str(tar_guidance_scale).replace('.', '')}_v1"
OUTPUT_ROOT = f"outputs/{EXPERIMENT_TAG}"
PIPELINE_LOAD_MODE = "sequential_cpu_offload"

sample_limit_choice = 20  # Use 10 for a smoke test; 20 has more variety; "full" runs all pairs.
sample_limit = None if str(sample_limit_choice).lower() == "full" else int(sample_limit_choice)
sample_offset = 0
image_resolution = 512

force_rerun = False
force_metrics = True
RUN_MODEL_PREFLIGHT = False
NUM_QUALITATIVE_EXAMPLES = min(5, sample_limit or 5)

metrics_to_compute = ["CLIP-T", "CLIP-I", "LPIPS", "DINO", "DreamSim"]
metric_image_resolution = image_resolution
lpips_resize = image_resolution
skip_failed_metrics = True

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token, leave blank if not needed: ")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

print("Branch:", BRANCH)
print("Model:", model_name)
print("Methods:", methods_to_run)
print("Budget mode:", budget_mode)
print("Target CFG:", tar_guidance_scale)
print("Bridge settings:", bridge_settings)
print("Sample limit:", sample_limit if sample_limit is not None else "full")
print("Output root:", OUTPUT_ROOT)
print("Image resolution:", image_resolution if image_resolution else "native")
print("Pipeline load mode:", PIPELINE_LOAD_MODE)


In [ ]:
# Clone repository, install dependencies, and authenticate if needed.
import os
import subprocess
from pathlib import Path

if not Path(WORKDIR).exists():
    subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
subprocess.run(["git", "fetch", "origin"], check=False)
checkout = subprocess.run(["git", "checkout", BRANCH], text=True, capture_output=True)
if checkout.returncode != 0:
    print(checkout.stdout)
    print(checkout.stderr)
    raise RuntimeError("Could not checkout the evaluation branch. Push the branch first, then rerun this cell.")
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=False)
subprocess.run(["git", "status", "--short", "--branch"], check=False)

subprocess.run([
    "pip", "install", "-q", "--upgrade",
    "plotly==5.24.1",
    "matplotlib",
    "diffusers>=0.31.0",
    "transformers>=4.44.0",
    "accelerate>=0.33.0",
    "safetensors",
    "sentencepiece",
    "einops",
    "pyyaml",
    "huggingface_hub",
    "lpips",
    "dreamsim",
    "torchao>=0.16.0",
], check=True)

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face.")
else:
    print("No HF token provided. Public downloads only.")


## Load Model: SD3 or FLUX

In [ ]:
# Build the experiment YAML for the selected model and bridge settings.
import json
import subprocess
from pathlib import Path
import pandas as pd
import yaml

MODEL_ROOT = Path(OUTPUT_ROOT) / model_name
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
EXP_YAML = MODEL_ROOT / f"{model_name}_tuned_eval_config.yaml"
RUN_SUMMARY_CSV = MODEL_ROOT / "run_summary.csv"

cmd = [
    "python", "flowedit_eval.py", "write-config",
    "--model_name", model_name,
    "--dataset_yaml", DATASET_YAML,
    "--methods", ",".join(methods_to_run),
    "--budget_mode", budget_mode,
    "--tar_guidance_scale", str(tar_guidance_scale),
    "--bridge_settings_json", json.dumps(bridge_settings),
    "--output_yaml", str(EXP_YAML),
]
if src_guidance_scale is not None:
    cmd += ["--src_guidance_scale", str(src_guidance_scale)]

print("$", " ".join(cmd))
subprocess.run(cmd, check=True)

with open(EXP_YAML, "r", encoding="utf-8") as f:
    exp = yaml.safe_load(f)

config_table = pd.DataFrame(exp)
for col in ["pc_guidance_lambda", "pc_guidance_gamma", "pc_guidance_weight", "pc_enable_below_t"]:
    if col not in config_table:
        config_table[col] = ""

display(config_table[[
    "exp_name", "method_name", "setting_id", "solver_type", "model_type", "T_steps", "n_max",
    "src_guidance_scale", "tar_guidance_scale", "pc_guidance_lambda", "pc_guidance_gamma",
    "pc_guidance_weight", "pc_enable_below_t", "seed",
]])

if RUN_MODEL_PREFLIGHT:
    subprocess.run([
        "python", "run_script.py",
        "--exp_yaml", str(EXP_YAML),
        "--pipeline_load_mode", PIPELINE_LOAD_MODE,
        "--preflight_only",
    ], check=True)


## Load FlowEdit Dataset

In [ ]:
# Inspect the released FlowEdit dataset and selected sample subset.
from flowedit_eval import load_eval_samples
import pandas as pd

all_samples = load_eval_samples(DATASET_YAML)
selected_samples = load_eval_samples(
    DATASET_YAML,
    sample_limit=sample_limit,
    sample_offset=sample_offset,
)

print("Full dataset pairs:", len(all_samples))
print("Full dataset images:", len({s.image_id for s in all_samples}))
print("Selected pairs:", len(selected_samples))

display(pd.DataFrame([s.__dict__ for s in selected_samples]).head(20))


## Choose Sample Limit and Methods

In [ ]:
# Record the exact run choices before editing starts.
run_config = {
    "model_name": model_name,
    "methods_to_run": methods_to_run,
    "budget_mode": budget_mode,
    "src_guidance_scale": src_guidance_scale if src_guidance_scale is not None else "model_default",
    "tar_guidance_scale": tar_guidance_scale,
    "bridge_settings": json.dumps(bridge_settings),
    "dataset_yaml": DATASET_YAML,
    "sample_limit": sample_limit if sample_limit is not None else "full",
    "sample_offset": sample_offset,
    "image_resolution": image_resolution if image_resolution else "native",
    "pipeline_load_mode": PIPELINE_LOAD_MODE,
    "force_rerun": force_rerun,
    "force_metrics": force_metrics,
    "output_root": OUTPUT_ROOT,
}
display(pd.DataFrame([run_config]))


## Run Editing

In [ ]:
# Generate edited images. Existing images are reused only if their metadata matches the current config.
import subprocess
import pandas as pd

cmd = [
    "python", "-u", "run_script.py",
    "--device_number", "0",
    "--exp_yaml", str(EXP_YAML),
    "--dataset_yaml", DATASET_YAML,
    "--eval_output_root", OUTPUT_ROOT,
    "--run_summary_csv", str(RUN_SUMMARY_CSV),
    "--pipeline_load_mode", PIPELINE_LOAD_MODE,
    "--sample_offset", str(sample_offset),
]
if sample_limit is not None:
    cmd += ["--sample_limit", str(sample_limit)]
if image_resolution:
    cmd += ["--image_resolution", str(image_resolution)]
if force_rerun:
    cmd += ["--force_rerun"]

print("$", " ".join(cmd))
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"run_script.py failed with exit code {return_code}. If it was -9/SIGKILL, reduce sample_limit or keep image_resolution=512.")

run_summary = pd.read_csv(RUN_SUMMARY_CSV)
cols = [
    "sample_id", "method", "setting_id", "solver_type", "image_resolution", "actual_nfe",
    "elapsed_seconds", "cached_generation", "output_image",
]
display(run_summary[cols].head(40))
print("Rows:", len(run_summary))

nfe_check = run_summary.groupby(["method", "setting_id"], dropna=False).agg(
    num_samples=("sample_id", "count"),
    mean_nfe=("actual_nfe", "mean"),
    cached=("cached_generation", "sum"),
    mean_runtime=("elapsed_seconds", "mean"),
).reset_index()
display(nfe_check)


## Compute Metrics

In [ ]:
# Compute FlowEdit paper metrics. DreamSim is optional and will be left blank if the Colab package stack cannot load it.
import subprocess

metrics_cmd = [
    "python", "-u", "flowedit_eval.py", "metrics",
    "--run_summary_csv", str(RUN_SUMMARY_CSV),
    "--output_root", OUTPUT_ROOT,
    "--model_name", model_name,
    "--metrics", ",".join(metrics_to_compute),
    "--metric_image_resolution", str(metric_image_resolution),
    "--lpips_resize", str(lpips_resize),
]
if skip_failed_metrics:
    metrics_cmd += ["--skip_failed_metrics"]
if force_metrics:
    metrics_cmd += ["--force_metrics"]

print("$", " ".join(metrics_cmd))
process = subprocess.Popen(metrics_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"flowedit_eval.py metrics failed with exit code {return_code}.")


## Generate Plots and Tables

In [ ]:
# Display summary, delta, win-rate tables, and the CLIP-T vs LPIPS plot.
from IPython.display import Image as IPImage, display
import pandas as pd
from pathlib import Path

SUMMARY_CSV = MODEL_ROOT / "summary_metrics.csv"
DELTA_CSV = MODEL_ROOT / "delta_vs_flowedit_baseline.csv"
WIN_RATE_CSV = MODEL_ROOT / "win_rate_vs_flowedit_baseline.csv"
PLOT_PNG = MODEL_ROOT / "plots" / "clip_t_vs_lpips.png"

summary = pd.read_csv(SUMMARY_CSV)
delta = pd.read_csv(DELTA_CSV)
win_rate = pd.read_csv(WIN_RATE_CSV)

print("Summary metrics")
display(summary)
print("Delta vs FlowEdit baseline. Positive means improvement under each metric direction.")
display(delta)
print("Win rate vs FlowEdit baseline")
display(win_rate)

score = summary.copy()
for col in [
    "CLIP-T_delta_vs_flowedit_baseline",
    "LPIPS_delta_vs_flowedit_baseline",
    "DINO_delta_vs_flowedit_baseline",
]:
    score[col] = pd.to_numeric(score[col], errors="coerce")
score = score[score["method"] != "flowedit_baseline"].copy()
score["structure_delta_mean"] = score[["LPIPS_delta_vs_flowedit_baseline", "DINO_delta_vs_flowedit_baseline"]].mean(axis=1)
score["balanced_delta"] = score[["CLIP-T_delta_vs_flowedit_baseline", "LPIPS_delta_vs_flowedit_baseline", "DINO_delta_vs_flowedit_baseline"]].mean(axis=1)
print("Bridge ranking helper")
display(score.sort_values(["balanced_delta", "CLIP-T_delta_vs_flowedit_baseline"], ascending=False))

if PLOT_PNG.exists():
    display(IPImage(filename=str(PLOT_PNG)))
else:
    print("Plot not found:", PLOT_PNG)


## Display Qualitative Comparison Examples

In [ ]:
# Show source and edited images side by side for a few selected pairs and settings.
import base64
import html
from pathlib import Path
import pandas as pd
from IPython.display import HTML, display

metrics = pd.read_csv(MODEL_ROOT / "all_metrics_per_pair.csv")
summary = pd.read_csv(MODEL_ROOT / "summary_metrics.csv")
examples = list(dict.fromkeys(metrics["sample_id"].tolist()))[:NUM_QUALITATIVE_EXAMPLES]

def group_label(method, setting_id):
    return method if str(setting_id) in {"", "default", "nan"} else f"{method}/{setting_id}"

groups = []
for _, row in summary.iterrows():
    groups.append((row["method"], row.get("setting_id", "default"), group_label(row["method"], row.get("setting_id", "default"))))

def image_to_data_uri(path):
    path = Path(path)
    suffix = path.suffix.lower().replace(".", "") or "png"
    if suffix == "jpg":
        suffix = "jpeg"
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:image/{suffix};base64,{data}"

def img_tag(path, width=155):
    return f"<img src='{image_to_data_uri(path)}' style='width:{width}px;max-width:100%;border:1px solid #ddd;'>"

rows_html = []
for sample_id in examples:
    sample_rows = metrics[metrics["sample_id"] == sample_id]
    first = sample_rows.iloc[0]
    cells = [
        "<td>" +
        f"<b>{html.escape(sample_id)}</b><br>" +
        img_tag(first["source_image_path"]) +
        f"<br><small>{html.escape(str(first['target_prompt'])[:180])}</small>" +
        "</td>"
    ]
    for method, setting_id, label in groups:
        row_df = sample_rows[(sample_rows["method"] == method) & (sample_rows["setting_id"].astype(str) == str(setting_id))]
        if row_df.empty:
            cells.append(f"<td><b>{html.escape(label)}</b><br><em>missing</em></td>")
            continue
        row = row_df.iloc[0]
        metric_line = (
            f"CLIP-T {float(row['CLIP-T']):.3f} | "
            f"LPIPS {float(row['LPIPS']):.3f} | "
            f"DINO {float(row['DINO']):.3f}"
        )
        cells.append(
            "<td>" +
            f"<b>{html.escape(label)}</b><br>" +
            img_tag(row["output_image"]) +
            f"<br><small>{html.escape(metric_line)}</small>" +
            "</td>"
        )
    rows_html.append("<tr>" + "".join(cells) + "</tr>")

display(HTML("<table>" + "".join(rows_html) + "</table>"))
